# T3P Packet Inspector

In [3]:
import struct
import re
import os
import mmap

def inspect_t3p_range(filepath, start_packet, end_packet):
    print(f"--- T3P EXACT PACKET INSPECTOR ---")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Printing records #{start_packet} through #{end_packet}...")
    print(f"Scanning the entire file for global totals (this may take a moment for massive files)...\n")
    
    text_log_pattern = re.compile(rb'(?:\d+\t)+\d+\r?\n')
    
    with open(filepath, 'rb') as f:
        # Use memory-mapping to scan massive files instantly without overloading RAM
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Pre-scan for all telemetry text injections
        telemetry_traps = list(text_log_pattern.finditer(mm))
        
        offset = 0
        record_count = 0
        header_printed = False
        
        # Global Counters for the entire file
        non_zero_overflow_count = 0
        total_photon_hits = 0
        total_telemetry_packets = 0
        
        # Loop over the ENTIRE file to get true global totals
        while offset < len(mm):
            
            # 1. CHECK FOR TELEMETRY INJECTION INTERRUPTIONS
            if telemetry_traps and offset <= telemetry_traps[0].start() < offset + 16:
                trap = telemetry_traps.pop(0)
                
                # Jump over any corrupted/cut-off bytes directly to the text
                offset = trap.start()
                
                # The text string itself counts as one record in the sequence
                record_count += 1
                total_telemetry_packets += 1
                
                text_bytes = mm[trap.start() : trap.end()]
                
                if start_packet <= record_count <= end_packet:
                    text_str = text_bytes.decode('ascii', errors='ignore').strip().replace('\t', ' \\t ')
                    print(f"\n[{record_count:08d}] [OFFSET: {trap.start():08X}] 🖂 TELEMETRY STRING | Length: {len(text_bytes):02d} bytes | Text: '{text_str}'\n")
                    header_printed = False # Reset header so it reprints after the text interruption
                
                # Resume standard 16-byte reading immediately after the text ends
                offset = trap.end()
                continue
                
            # 2. READ THE STANDARD 16-BYTE PACKET
            if offset + 16 > len(mm):
                break 
                
            packet = mm[offset : offset + 16]
            
            # Unpack all 5 variables from the C-Struct
            matrixIdx, toa, overflow, ftoa, tot = struct.unpack('<IQBBH', packet)
            record_count += 1
            
            # Evaluate packet type
            is_hw_trigger = (overflow == 10 and matrixIdx == 0)
            
            # Global Tally logic
            if not is_hw_trigger:
                total_photon_hits += 1
                if overflow != 0:
                    non_zero_overflow_count += 1
            
            # 3. PRINT THE PACKET ONLY IF IT FALLS IN THE RANGE
            if start_packet <= record_count <= end_packet:
                
                if not header_printed:
                    print(f"{'RECORD #':<10} | {'BYTE OFFSET':<11} | {'PACKET TYPE':<14} | {'matrixIdx':<10} | {'ToA':<12} | {'ToT':<5} | {'fToA':<4} | {'Overflow'}")
                    print("-" * 95)
                    header_printed = True
                
                if is_hw_trigger:
                    ptype = "⚡ HW TRIGGER"
                else:
                    ptype = "🔵 PHOTON HIT"
                    
                print(f"[{record_count:08d}] | {offset:08X}    | {ptype:<14} | {matrixIdx:<10} | {toa:<12} | {tot:<5} | {ftoa:<4} | {overflow}")
                
            offset += 16
            
        mm.close()
        
    print("\n--- INSPECTION COMPLETE ---")
    print(f"► Total telemetry packets (Entire file): {total_telemetry_packets}")
    print(f"► Total photon hits (Entire file):       {total_photon_hits}")
    print(f"► Photon hits with overflow != 0:        {non_zero_overflow_count}")

# ==========================================
# EXECUTION
# ==========================================
t3p_file = r"G:\האחסון שלי\X-Ray-IFM\Test Files\Sync_test\sync_test_21.5.t3p"

# Change these two numbers to whatever range you want to print!
start = 0
end = 500

inspect_t3p_range(t3p_file, start, end)

--- T3P EXACT PACKET INSPECTOR ---
File: sync_test_21.5.t3p
Printing records #0 through #500...
Scanning the entire file for global totals (this may take a moment for massive files)...


[00000001] [OFFSET: 00000000] 🖂 TELEMETRY STRING | Length: 14 bytes | Text: '0 \t 0 \t 62 \t 0 \t 0 \t 10'


[00000002] [OFFSET: 0000000E] 🖂 TELEMETRY STRING | Length: 18 bytes | Text: '1 \t 0 \t 62 \t 0 \t 49152 \t 10'

RECORD #   | BYTE OFFSET | PACKET TYPE    | matrixIdx  | ToA          | ToT   | fToA | Overflow
-----------------------------------------------------------------------------------------------
[00000003] | 00000020    | 🔵 PHOTON HIT   | 7434       | 132366       | 12    | 17   | 0
[00000004] | 00000030    | 🔵 PHOTON HIT   | 7691       | 132366       | 16    | 18   | 0
[00000005] | 00000040    | 🔵 PHOTON HIT   | 7690       | 132366       | 76    | 18   | 0
[00000006] | 00000050    | 🔵 PHOTON HIT   | 2543       | 167669       | 9     | 15   | 0
[00000007] | 00000060    | 🔵 PHOTON HIT   | 

# .h5 Bins Inspector

In [ ]:
import h5py
import numpy as np
import os

def inspect_px5_h5(filepath, start_bin, end_bin):
    print(f"--- HDF5 PX5 DATA INSPECTOR ---")
    print(f"File: {os.path.basename(filepath)}")

    try:
        with h5py.File(filepath, 'r') as f:
            # 1. Read the Metadata attributes written by the DAQ script
            bin_s = f.attrs['bin_s']
            
            # 2. Access the main dataset
            if 'px5CountsPerBin' not in f:
                print("Error: Dataset 'px5CountsPerBin' not found in file.")
                return
                
            dset = f['px5CountsPerBin']
            total_bins = dset.shape[0]
            
            print(f"Total Bins   : {total_bins:,}")
            print(f"Bin Resolution: {bin_s * 1e6:.1f} µs ({bin_s} s)")
            print(f"File Duration: {total_bins * bin_s:.2f} seconds")
            
            # We can sum the entire array instantly to find the total hits
            print(f"Total Photons: {np.sum(dset):,}") 
            print("-" * 55)
            
            # 3. Validate user input range
            if start_bin < 0: start_bin = 0
            if end_bin >= total_bins: end_bin = total_bins - 1
            if start_bin > end_bin:
                print("Invalid range selected.")
                return
            
            print(f"Scanning bins #{start_bin} to #{end_bin}...\n")
            print(f"{'BIN INDEX':<12} | {'TIME (Seconds)':<15} | {'PHOTON COUNT'}")
            print("-" * 45)
            
            # 4. Extract only the specific slice of memory requested
            counts_in_range = dset[start_bin : end_bin + 1]
            
            for i, count in enumerate(counts_in_range):
                actual_bin = start_bin + i
                time_s = actual_bin * bin_s
                
                # Add a visual flag if a photon was actually detected in this bin
                if count > 0:
                    count_str = f"{count}  <-- 🟢 HIT"
                else:
                    count_str = str(count)
                    
                print(f"[{actual_bin:08d}]   | {time_s:<15.6f} | {count_str}")
                
    except Exception as e:
        print(f"Failed to read HDF5 file: {e}")

# ==========================================
# EXECUTION
# ==========================================
# Point this to your generated DAQ file
h5_file = r"G:\האחסון שלי\ESRF IFM\esrf_px5_data_000.h5" 

# Select the range of bins you want to look at
start = 0
end   = 1000

inspect_px5_h5(h5_file, start, end)